# RAV-Med: Step 6e, cost comparison (ERS vs LLM judge)
**Question.** How much does one ERS check cost compared with one zero-shot 7B judge call, on the same GPU and the same passages?

**Fixed before running.** 300 (claim, passage) rows drawn with seed 42 from `rav_med_full_results_N1000.csv`; same models, batch sizes and 250-word truncation as Step 6a (ERS: SBERT bs 64 + RoBERTa-large-MNLI bs 32; judge: Qwen2.5-7B-Instruct 4-bit, batch 8, one forward pass, one prompt). Two warm-up batches are discarded; 3 timed repeats; median is reported. Nothing is tuned. Report whatever comes out.

Run on a **T4 GPU** (the same kind as Steps 6a/6b). Upload `rav_med_full_results_N1000.csv` when asked. About 10 minutes.
**Reported:** milliseconds per passage, passages per minute, peak GPU memory, parameter counts, and the ERS-on-CPU cost.

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes pandas numpy

In [ ]:
import os, time, json, statistics, shutil
import numpy as np, pandas as pd, torch
DRY_RUN = False
SEED, N, WARMUP, REPEATS = 42, 300, 2, 3
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"; NLI_MODEL = "roberta-large-mnli"; LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_WORDS, JBATCH, EBATCH, NBATCH = 250, 8, 64, 32
OUT = "step6e_outputs"; os.makedirs(OUT, exist_ok=True)
if not DRY_RUN: assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU (Runtime > Change runtime type)."
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"; print("GPU:", GPU)
if not os.path.exists("rav_med_full_results_N1000.csv"):
    from google.colab import files; print("Upload rav_med_full_results_N1000.csv"); files.upload()
df = pd.read_csv("rav_med_full_results_N1000.csv").sample(n=N, random_state=SEED).reset_index(drop=True)
claims, passages = df.claim.tolist(), df.passage.tolist()
def trunc(t, n=MAX_WORDS):
    w = t.split(); return t if len(w) <= n else " ".join(w[:n]) + " ..."
print(N, "rows; mean passage words:", round(np.mean([len(p.split()) for p in passages]), 1))
def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()
def timed(fn, items, bs):
    """seconds per item: median over REPEATS full passes, after WARMUP discarded batches"""
    for i in range(WARMUP): fn(items[i*bs:(i+1)*bs])
    ts = []
    for _ in range(REPEATS):
        sync(); t0 = time.perf_counter()
        for i in range(0, len(items), bs): fn(items[i:i+bs])
        sync(); ts.append((time.perf_counter() - t0) / len(items))
    return statistics.median(ts), ts
rows = []

## 1. ERS on GPU (similarity + contradiction, exactly as in the paper)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=dev)
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL); nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(dev).eval()
nparam = lambda m: sum(p.numel() for p in m.parameters())
P_ERS = nparam(embedder) + nparam(nli); print("ERS parameters (M):", round(P_ERS/1e6, 1))
@torch.no_grad()
def ers_batch(pairs):
    cl = [c for c, p in pairs]; ps = [p for c, p in pairs]
    ec = embedder.encode(cl, batch_size=EBATCH, convert_to_numpy=True, normalize_embeddings=True)
    ep = embedder.encode(ps, batch_size=EBATCH, convert_to_numpy=True, normalize_embeddings=True)
    S = (ec * ep).sum(1)
    enc = nli_tok(ps, cl, truncation=True, padding=True, max_length=256, return_tensors="pt").to(dev)
    C = torch.softmax(nli(**enc).logits.float(), -1)[:, nli.config.label2id.get("CONTRADICTION", 0)].cpu().numpy()
    return S - C
items = list(zip(claims, passages))
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
t, ts = timed(ers_batch, items, NBATCH)
mem = torch.cuda.max_memory_allocated()/2**30 if torch.cuda.is_available() else float("nan")
rows.append(dict(method="ERS (SBERT + RoBERTa-large-MNLI)", device=GPU, params_M=round(P_ERS/1e6,1), ms_per_passage=t*1000, passages_per_min=60/t, peak_gpu_GiB=mem, repeats_ms=[round(x*1000,2) for x in ts]))
print(rows[-1])

## 2. ERS on CPU (no GPU needed at inference)

In [ ]:
embedder.to("cpu"); nli.to("cpu"); dev = "cpu"
sub = items[:60]
t, ts = timed(ers_batch, sub, 16)
import platform; cpu = platform.processor() or platform.machine()
rows.append(dict(method="ERS on CPU (60 passages)", device=f"CPU ({os.cpu_count()} cores)", params_M=round(P_ERS/1e6,1), ms_per_passage=t*1000, passages_per_min=60/t, peak_gpu_GiB=0.0, repeats_ms=[round(x*1000,2) for x in ts]))
print(rows[-1])
del embedder, nli
import gc; gc.collect(); torch.cuda.empty_cache()

## 3. Zero-shot 7B judge (one prompt per passage, same prompt text as Step 6a)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, device_map={"": 0}, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")).eval()
P_LLM = nparam(llm); print("judge parameters (M, as stored; 4-bit weights are packed):", round(P_LLM/1e6, 1))
YES, NO = tok.encode("Yes", add_special_tokens=False)[0], tok.encode("No", add_special_tokens=False)[0]
SYS = "You are a careful medical evidence reviewer. Answer with a single word: Yes or No."
Q = {"rel": "Is the passage relevant to the claim, meaning it addresses the same medical question?",
     "trust": "Would a careful clinician treat this passage as reliable evidence for the claim, that is, relevant to it and consistent with it?"}
def chat(s, u): return tok.apply_chat_template([{"role": "system", "content": s}, {"role": "user", "content": u}], tokenize=False, add_generation_prompt=True)
@torch.no_grad()
def judge_batch(prompts):
    enc = tok([chat(SYS, u) for u in prompts], return_tensors="pt", padding=True).to("cuda")
    lg = llm(**enc).logits[:, -1, :].float()[:, [YES, NO]]
    return torch.softmax(lg, -1)[:, 0].cpu().numpy()
torch.cuda.reset_peak_memory_stats()
for k in ["trust"]:
    prompts = [f"Claim: {c}\n\nPassage: {trunc(p)}\n\n{Q[k]} Answer Yes or No." for c, p in items]
    t, ts = timed(judge_batch, prompts, JBATCH)
    rows.append(dict(method=f"LLM judge, 1 prompt ({k}), Qwen2.5-7B 4-bit", device=GPU, params_M=round(P_LLM/1e6,1), ms_per_passage=t*1000, passages_per_min=60/t, peak_gpu_GiB=torch.cuda.max_memory_allocated()/2**30, repeats_ms=[round(x*1000,2) for x in ts]))
    print(rows[-1])
R = pd.DataFrame(rows)
ers = R.loc[0, "ms_per_passage"]; R["cost_relative_to_ERS_gpu"] = R.ms_per_passage / ers
R["note"] = ""; R.loc[R.method.str.startswith("LLM judge"), "note"] = "The 4-judge set used for the 'relevance minus contradiction' scores costs 2-4x this; ERS needs no extra pass."
print(R.drop(columns=["repeats_ms"]).round(2).to_string(index=False))
R.to_csv(f"{OUT}/s6e_cost_timing.csv", index=False)
json.dump(dict(gpu=GPU, n=N, seed=SEED, warmup=WARMUP, repeats=REPEATS, max_words=MAX_WORDS, jbatch=JBATCH), open(f"{OUT}/s6e_settings.json", "w"))
shutil.make_archive("rav_med_step6e_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("rav_med_step6e_outputs.zip")
except ImportError: pass
print("Saved rav_med_step6e_outputs.zip")